# Sumarização e avaliação de desempenho de diálogos de atendimento ao cliente pelo Twitter

- Constrói dataset contendo posts do Twitter e sumarização humana (ground truth)

## Define constantes

In [1]:
PATH_PREPARED_DATASET = f'../data/interim/dataset_summarization.csv'

PATH_DATASET_TWEETS = '../data/raw/kaggle/twcs.csv'

PATH_DATASET_SUMMARIES = '../data/raw/github/final_train_tweetsum.jsonl'

## Carrega bibliotecas

In [ ]:
import os

from rich import print
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Prepara dataset

### Messages dataset - [Customer Support on Twitter (Kaggle)](https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter)

In [3]:
df_tweets = pd.read_csv(PATH_DATASET_TWEETS)
print(df_tweets.shape)
df_tweets.head()

(2811774, 7)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


### Annotations dataset - https://github.com/guyfe/Tweetsumm

In [4]:
df_summ = pd.read_json(PATH_DATASET_SUMMARIES, lines=True)
print(df_summ.shape)
df_summ.head()

(879, 3)

,conversation_id,tweet_ids_sentence_offset,annotations
0,b065262210783596c1fe79466b8f8985,"[{'tweet_id': 87076, 'sentence_offsets': ['[0,...","[{'extractive': [{'tweet_id': 87076, 'sentence..."
1,1e1d8fd4f95c984fb78687c9e946dc97,"[{'tweet_id': 607297, 'sentence_offsets': ['[0...","[{'extractive': [{'tweet_id': 607297, 'sentenc..."
2,b5773077fa55c381260390472deff4c2,"[{'tweet_id': 428556, 'sentence_offsets': ['[0...","[{'extractive': [{'tweet_id': 739293, 'sentenc..."
3,3574d6b4418cb546a90d1561bacd66a2,"[{'tweet_id': 587942, 'sentence_offsets': ['[0...","[{'extractive': [{'tweet_id': 587942, 'sentenc..."
4,f30d2bbc15e7ff32244761b38b67d160,"[{'tweet_id': 861415, 'sentence_offsets': ['[0...","[{'extractive': [{'tweet_id': 861405, 'sentenc..."


In [5]:
print(df_summ.iloc[0].annotations)

[
    {
        'extractive': [
            {'tweet_id': 87076, 'sentence_offset': '[0, 140]'},
            {'tweet_id': 87074, 'sentence_offset': '[41, 139]'},
            {'tweet_id': 87073, 'sentence_offset': '[0, 61]'}
        ],
        'abstractive': [
            'Customer enquired about his Iphone and Apple watch which is not showing his any steps/activity and 
health activities.',
            'Agent is asking to move to DM and look into it.'
        ]
    },
    {
        'extractive': [
            {'tweet_id': 87076, 'sentence_offset': '[0, 140]'},
            {'tweet_id': 87074, 'sentence_offset': '[41, 139]'}
        ],
        'abstractive': [
            'The customer has a problem.',
            'The agent in a very professional way tries to help the client.'
        ]
    },
    {
        'extractive': [
            {'tweet_id': 87076, 'sentence_offset': '[0, 140]'},
            {'tweet_id': 87072, 'sentence_offset': '[19, 87]'},
            {'tweet_id': 87069, 'sentence_offset': '[0, 72]'},
            {'tweet_id': 87068, 'sentence_offset': '[0, 55]'}
        ],
        'abstractive': [
            'Health and activity functions are not working with the smartwatch and phone.',
            'Asks if the customer had restarted the items, offers to take this to DM to help resolve the issue.'
        ]
    }
]

In [6]:
def sort_tweet_text(row):
    sorted_ids = sorted(row['created_at_list'], key=lambda x: row['created_at_list'][x])
    sorted_created_at = {idx: row['created_at_list'][idx] for idx in sorted_ids}
    tweet_text_sorted = [row['tweet_text'][idx] for idx in sorted_created_at.keys()]
    return tweet_text_sorted

In [7]:
df_exp = df_summ.explode('annotations')
df_exp['extractive'] = df_exp['annotations'].apply(lambda x: x['extractive'])
df_exp['abstractive'] = df_exp['annotations'].apply(lambda x: x['abstractive'])

df_exp = df_exp.dropna()
df_exp = df_exp[~df_exp.annotations.duplicated()]

df_exp['tweet_ids'] = df_exp['extractive'].apply(
    lambda x: sorted([y['tweet_id'] for y in x])
)

df_indexed = df_tweets.set_index('tweet_id')

df_exp['tweet_text'] = df_exp['tweet_ids'].apply(
    lambda x: {k: v for k, v in zip(x, df_indexed.loc[x, 'text'].to_list())}
)

df_exp['created_at_list'] = df_exp['tweet_ids'].apply(
    lambda x: {k: v for k, v in zip(x, pd.to_datetime(df_indexed.loc[x, 'created_at']).tolist())}
)

df_exp = df_exp[~df_exp.tweet_text.duplicated()]

df_exp[['tweet_ids', 'tweet_text', 'created_at_list']].head()

print(df_exp[['tweet_text', 'created_at_list']].iloc[2].to_dict())

{
    'tweet_text': {
        87068: '@135060 Let’s move to DM and look into this a bit more. When reaching out in DM, let us know when 
this first started happening please. For example, did it start after an update or after installing a certain app? 
https://t.co/GDrqU22YpT',
        87069: '@AppleSupport Yes, everything seems fine, it’s just Health and activity.',
        87072: '@135060 Thank you. Have you tried restarting both devices since this started happening?',
        87076: 'So neither my iPhone nor my Apple Watch are recording my steps/activity, and Health doesn’t 
recognise either source anymore for some reason. Any ideas? https://t.co/m9DPQbkftD'
    },
    'created_at_list': {
        87068: Timestamp('2017-11-30 14:29:00+0000', tz='UTC'),
        87069: Timestamp('2017-11-30 08:46:32+0000', tz='UTC'),
        87072: Timestamp('2017-11-28 22:13:56+0000', tz='UTC'),
        87076: Timestamp('2017-11-28 20:45:52+0000', tz='UTC')
    }
}

In [8]:
df_exp['tweet_text_sorted']= df_exp[['tweet_ids', 'tweet_text', 'created_at_list']].apply(
    sort_tweet_text,
    axis=1
)
print(df_exp[['created_at_list', 'tweet_text_sorted']].iloc[4].to_dict())

df_exp['human_summary'] = df_exp['abstractive'].apply(lambda x: ' '.join(x))

df_exp.drop(columns=['tweet_ids_sentence_offset', 'annotations', 'abstractive', 'extractive', 'tweet_text'], inplace=True)
df_exp['tweet_text_sorted'] = df_exp['tweet_text_sorted'].apply(lambda x: ' '.join(x))
df_exp.rename(columns={'tweet_text_sorted': 'tweet_texts'}, inplace=True)
df_exp.head()

{
    'created_at_list': {
        739293: Timestamp('2017-10-18 15:22:17+0000', tz='UTC'),
        739295: Timestamp('2017-10-18 15:38:19+0000', tz='UTC')
    },
    'tweet_text_sorted': [
        "@AskAmex Signed up for new card with Delta to book immediately book tix. Card number didn't come up. 
Customer svce refused to help.",
        "@216929 Good morning, thanks for reaching out. Please call our New Accounts Team at 877-399-3086, for 
assistance. They're available,"
    ]
}

,conversation_id,tweet_ids,created_at_list,tweet_texts,human_summary
0,b065262210783596c1fe79466b8f8985,"[87073, 87074, 87076]","{87073: 2017-11-28 21:57:00+00:00, 87074: 2017...",So neither my iPhone nor my Apple Watch are re...,Customer enquired about his Iphone and Apple w...
0,b065262210783596c1fe79466b8f8985,"[87074, 87076]","{87074: 2017-11-28 21:11:57+00:00, 87076: 2017...",So neither my iPhone nor my Apple Watch are re...,The customer has a problem. The agent in a ver...
0,b065262210783596c1fe79466b8f8985,"[87068, 87069, 87072, 87076]","{87068: 2017-11-30 14:29:00+00:00, 87069: 2017...",So neither my iPhone nor my Apple Watch are re...,Health and activity functions are not working ...
1,1e1d8fd4f95c984fb78687c9e946dc97,"[607293, 607296, 607297, 607297]","{607293: 2017-11-22 12:16:27+00:00, 607296: 20...",@115850 hi team! i m planning to get Apple Air...,Customer is eager to know about the replacemen...
2,b5773077fa55c381260390472deff4c2,"[739293, 739295]","{739293: 2017-10-18 15:22:17+00:00, 739295: 20...",@AskAmex Signed up for new card with Delta to ...,Signed up for an AmexCard with Delta but it di...


In [9]:
print(df_exp.iloc[0].to_dict())

{
    'conversation_id': 'b065262210783596c1fe79466b8f8985',
    'tweet_ids': [87073, 87074, 87076],
    'created_at_list': {
        87073: Timestamp('2017-11-28 21:57:00+0000', tz='UTC'),
        87074: Timestamp('2017-11-28 21:11:57+0000', tz='UTC'),
        87076: Timestamp('2017-11-28 20:45:52+0000', tz='UTC')
    },
    'tweet_texts': 'So neither my iPhone nor my Apple Watch are recording my steps/activity, and Health doesn’t 
recognise either source anymore for some reason. Any ideas? https://t.co/m9DPQbkftD @135060 Let’s investigate this 
together. To start, can you tell us the software versions your iPhone and Apple Watch are running currently? 
@AppleSupport My iPhone is on 11.1.2, and my watch is on 4.1.',
    'human_summary': 'Customer enquired about his Iphone and Apple watch which is not showing his any 
steps/activity and health activities. Agent is asking to move to DM and look into it.'
}

In [10]:
def calc_period(x):
    date_list = pd.to_datetime(df_indexed.loc[x, 'created_at']).tolist()
    return max(date_list) - min(date_list) 

In [11]:
df_exp['elapsed_time'] = df_exp['tweet_ids'].apply(
    calc_period
)

print(df_exp.iloc[0].to_dict())

{
    'conversation_id': 'b065262210783596c1fe79466b8f8985',
    'tweet_ids': [87073, 87074, 87076],
    'created_at_list': {
        87073: Timestamp('2017-11-28 21:57:00+0000', tz='UTC'),
        87074: Timestamp('2017-11-28 21:11:57+0000', tz='UTC'),
        87076: Timestamp('2017-11-28 20:45:52+0000', tz='UTC')
    },
    'tweet_texts': 'So neither my iPhone nor my Apple Watch are recording my steps/activity, and Health doesn’t 
recognise either source anymore for some reason. Any ideas? https://t.co/m9DPQbkftD @135060 Let’s investigate this 
together. To start, can you tell us the software versions your iPhone and Apple Watch are running currently? 
@AppleSupport My iPhone is on 11.1.2, and my watch is on 4.1.',
    'human_summary': 'Customer enquired about his Iphone and Apple watch which is not showing his any 
steps/activity and health activities. Agent is asking to move to DM and look into it.',
    'elapsed_time': Timedelta('0 days 01:11:08')
}

## Exporta dataset

In [12]:
df_exp.to_csv(PATH_PREPARED_DATASET, index=None)